{
 "cells": [
  {
   "cell_type": "markdown",
   "source": [
    "# Traffic Sentinel - Traffic Analysis\n",
    "**Uganda Focus** - Analyzing local video data for risk prediction\n",
    "\n",
    "Author: Keith Ndiema Kissa  \n",
    "Date: May 2026"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "## 1. Setup & Import Libraries"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "import os\n",
    "import cv2\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import json\n",
    "from datetime import datetime\n",
    "\n",
    "print('✅ Libraries imported successfully')"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "## 2. Define Paths (Using your existing folders)"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "INPUT_VIDEO_DIR = '../data/input_video'\n",
    "OUTPUT_DIR = '../data/output_results'\n",
    "SAMPLE_FRAMES_DIR = '../data/sample_framing'\n",
    "\n",
    "os.makedirs(OUTPUT_DIR, exist_ok=True)\n",
    "os.makedirs(SAMPLE_FRAMES_DIR, exist_ok=True)\n",
    "\n",
    "print('Input videos found:', os.listdir(INPUT_VIDEO_DIR) if os.path.exists(INPUT_VIDEO_DIR) else 'Folder not found')"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "## 3. Video Analysis Function"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "def analyze_video(video_path):\n",
    "    cap = cv2.VideoCapture(video_path)\n",
    "    if not cap.isOpened():\n",
    "        return None\n",
    "    \n",
    "    fps = int(cap.get(cv2.CAP_PROP_FPS))\n",
    "    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n",
    "    \n",
    "    vehicle_counts = []\n",
    "    frame_num = 0\n",
    "    \n",
    "    while cap.isOpened():\n",
    "        ret, frame = cap.read()\n",
    "        if not ret:\n",
    "            break\n",
    "        frame_num += 1\n",
    "        \n",
    "        if frame_num % 30 == 0:  # Sample every 30 frames\n",
    "            # Placeholder for vehicle detection\n",
    "            vehicle_count = np.random.randint(8, 30)  # Replace with real detection later\n",
    "            vehicle_counts.append(vehicle_count)\n",
    "            \n",
    "            # Save sample frame\n",
    "            if frame_num % 90 == 0:\n",
    "                frame_path = os.path.join(SAMPLE_FRAMES_DIR, f'sample_{os.path.basename(video_path)}_{frame_num}.jpg')\n",
    "                cv2.imwrite(frame_path, frame)\n",
    "    \n",
    "    cap.release()\n",
    "    \n",
    "    return {\n",
    "        'video_name': os.path.basename(video_path),\n",
    "        'duration_sec': round(frame_count / fps, 2) if fps > 0 else 0,\n",
    "        'total_frames': frame_count,\n",
    "        'avg_vehicles': round(np.mean(vehicle_counts), 2) if vehicle_counts else 0,\n",
    "        'max_vehicles': max(vehicle_counts) if vehicle_counts else 0\n",
    "    }"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "## 4. Run Analysis on All Videos"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "results = []\n",
    "\n",
    "if os.path.exists(INPUT_VIDEO_DIR):\n",
    "    videos = [f for f in os.listdir(INPUT_VIDEO_DIR) if f.endswith(('.mp4', '.avi', '.mov'))]\n",
    "    print(f'Found {len(videos)} videos to analyze')\n",
    "    \n",
    "    for video in videos:\n",
    "        video_path = os.path.join(INPUT_VIDEO_DIR, video)\n",
    "        result = analyze_video(video_path)\n",
    "        if result:\n",
    "            results.append(result)\n",
    "            print(f'✅ Analyzed: {video} | Avg Vehicles: {result[\"avg_vehicles\"]}')\n",
    "else:\n",
    "    print('Input video folder not found.')"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "## 5. Results & Visualization"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "df = pd.DataFrame(results)\n",
    "display(df)\n",
    "\n",
    "# Risk Level based on vehicle density\n",
    "df['risk_level'] = df['avg_vehicles'].apply(lambda x: 'High' if x > 18 else 'Medium' if x > 10 else 'Low')\n",
    "\n",
    "# Plot\n",
    "plt.figure(figsize=(10, 6))\n",
    "plt.bar(df['video_name'], df['avg_vehicles'], color='skyblue')\n",
    "plt.title('Average Vehicles Detected per Video')\n",
    "plt.xlabel('Video')\n",
    "plt.ylabel('Average Vehicle Count')\n",
    "plt.xticks(rotation=45)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "## 6. Save Results"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "output_file = os.path.join(OUTPUT_DIR, f'traffic_analysis_{datetime.now().strftime(\"%Y%m%d\")}.json')\n",
    "with open(output_file, 'w') as f:\n",
    "    json.dump(results, f, indent=2)\n",
    "\n",
    "print(f'Results saved to: {output_file}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}